In [2]:
#pip install numpy rasterio pytecplot

In [10]:
import tifffile as tiff
import numpy as np

data = tiff.memmap("../../ibcao_v5_1_2025_depth_400m.tif")  # does NOT load full file
ny, nx = data.shape

print(nx, ny)

14550 14550


In [11]:
# Downsampling factor
factor = 1

data_small = data[::factor, ::factor]

ny, nx = data_small.shape
print(f"New size: {nx} x {ny}")

with open("bathymetry.dat", "w") as f:
    f.write('TITLE = "Arctic Bathymetry"\n')
    f.write('VARIABLES = "X", "Y", "Depth"\n')
    f.write(f'ZONE I={nx}, J={ny}, F=POINT\n')

    for j in range(ny):
        for i in range(nx):
            f.write(f"{i} {j} {data_small[j, i]}\n")

print("Done!")

New size: 14550 x 14550
Done!


# Convert netcdf to tecplot format  

In [16]:
import tecio
from netCDF4 import Dataset
import numpy as np

nc = Dataset("../../gmm_clusters.nc")

In [17]:
nc.variables.keys()

dict_keys(['Latitude_[deg_N]', 'Longitude_[deg_E]', 'Depth_[m]', 'gmm_label', 'prob_cluster_0', 'prob_cluster_1', 'prob_cluster_2', 'prob_cluster_3', 'prob_cluster_4', 'prob_cluster_5', 'prob_cluster_6', 'prob_cluster_7', 'prob_cluster_8', 'index'])

In [27]:
lon = nc.variables['Longitude_[deg_E]'][:]
lat = nc.variables['Latitude_[deg_N]'][:]
depth = nc.variables['Depth_[m]'][:]
label = nc.variables['gmm_label'][:]
probs = [nc.variables[f'prob_cluster_{i}'][:] for i in range(9)]
N = len(depth)

In [26]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)

x, y = transformer.transform(lon, lat)

In [29]:
with open("gmm_animation.dat", "w") as f:
    f.write('TITLE = "GMM Animation"\n')
    f.write('VARIABLES = "X", "Y", "Depth", "Prob"\n')

    for k in range(9):
        f.write(f'ZONE T="Prob_{k}", I={N}, F=POINT\n')

        for i in range(N):
            f.write(f"{x[i]} {y[i]} {depth[i]} {probs[k][i]}\n")

print("Ready for animation in Tecplot")

Ready for animation in Tecplot
